In [8]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
import ipywidgets as widgets
from IPython.display import display, Audio, clear_output

# Create an output widget to hold the plots and audio so they don't overwrite the sliders
out = widgets.Output()

def update_simulation(g_Na, g_K, g_L, E_Na, E_K, E_L, i_inj_amp, C_m):
    with out:
        clear_output(wait=True)
        
        t = np.arange(0.0, 450.0, 0.01)

        # Kinetics
        def alpha_m(V): return 0.1 * (V + 40.0) / (1.0 - np.exp(-(V + 40.0) / 10.0))
        def beta_m(V): return 4.0 * np.exp(-(V + 65.0) / 18.0)
        def alpha_h(V): return 0.07 * np.exp(-(V + 65.0) / 20.0)
        def beta_h(V): return 1.0 / (1.0 + np.exp(-(V + 35.0) / 10.0))
        def alpha_n(V): return 0.01 * (V + 55.0) / (1.0 - np.exp(-(V + 55.0) / 10.0))
        def beta_n(V): return 0.125 * np.exp(-(V + 65) / 80.0)

        # Currents
        def I_Na(V, m, h): return g_Na * m**3 * h * (V - E_Na)
        def I_K(V, n): return g_K * n**4 * (V - E_K)
        def I_L(V): return g_L * (V - E_L)
        def I_inj(time): return i_inj_amp * (time > 100) - i_inj_amp * (time > 200) + (i_inj_amp * 3.5) * (time > 300) - (i_inj_amp * 3.5) * (time > 400)

        # Integration
        def dALLdt(X, time):
            V, m, h, n = X
            dVdt = (I_inj(time) - I_Na(V, m, h) - I_K(V, n) - I_L(V)) / C_m
            dmdt = alpha_m(V) * (1.0 - m) - beta_m(V) * m
            dhdt = alpha_h(V) * (1.0 - h) - beta_h(V) * h
            dndt = alpha_n(V) * (1.0 - n) - beta_n(V) * n
            return [dVdt, dmdt, dhdt, dndt]

        # Run ODE solver
        X = odeint(dALLdt, [-65, 0.05, 0.6, 0.32], t)
        V, m, h, n = X[:, 0], X[:, 1], X[:, 2], X[:, 3]
        
        ina, ik, il = I_Na(V, m, h), I_K(V, n), I_L(V)
        i_inj_values = [I_inj(time) for time in t]

        # --- Plotting ---
        plt.figure(figsize=(9, 10))

        plt.subplot(4, 1, 1)
        plt.title('Interactive Hodgkin-Huxley Neuron')
        plt.plot(t, V, 'k')
        plt.ylabel('Membrane Volts (mV)')

        plt.subplot(4, 1, 2)
        plt.plot(t, ina, 'c', label='Sodium ($I_{Na}$)')
        plt.plot(t, ik, 'y', label='Potassium ($I_{K}$)')
        plt.plot(t, il, 'm', label='Leak ($I_{L}$)')
        plt.ylabel('Current ($\\mu A/cm^2$)')
        plt.legend(loc='upper right')

        plt.subplot(4, 1, 3)
        plt.plot(t, m, 'r', label='m (Na+ Act.)')
        plt.plot(t, h, 'g', label='h (Na+ Inact.)')
        plt.plot(t, n, 'b', label='n (K+ Act.)')
        plt.ylabel('Gating Variable')
        plt.legend(loc='upper right')

        plt.subplot(4, 1, 4)
        plt.title('Stimulus / Injection Current')
        plt.plot(t, i_inj_values, 'k')
        plt.xlabel('Time (ms)')
        plt.ylabel('$I_{inj}$ ($\\mu A/cm^2$)')
        plt.ylim(min(i_inj_values)-5, max(i_inj_values)+5)

        plt.tight_layout()
        plt.show()

        # --- Sonification (Frequency Modulation) ---
        sr = 44100
        playback_duration = 3.0 
        t_audio = np.linspace(0, playback_duration, int(sr * playback_duration), endpoint=False)
        t_v_stretched = np.linspace(0, playback_duration, len(V))
        V_interp = np.interp(t_audio, t_v_stretched, V)
        
        V_scaled = (V_interp - (-80)) / (50 - (-80))
        V_scaled = np.clip(V_scaled, 0, 1)
        
        base_freq = 220.0
        inst_freq = base_freq * (2 ** (V_scaled * 4)) 
        phase = 2.0 * np.pi * np.cumsum(inst_freq) / sr
        audio_fm = np.sin(phase)

        print("Listen to the continuous voltage frequency:")
        display(Audio(data=audio_fm, rate=sr))

# --- UI Setup (Side by Side Layout) ---

# Define the 8 interactive parameters with explicit biological names
slider_layout = widgets.Layout(width='95%')
style = {'description_width': '180px'} # Ensure long names fit

w_g_Na = widgets.FloatSlider(value=120.0, min=0.0, max=250.0, step=10.0, description='Max Na+ Conductance ($g_{Na}$):', layout=slider_layout, style=style)
w_g_K = widgets.FloatSlider(value=36.0, min=0.0, max=100.0, step=2.0, description='Max K+ Conductance ($g_{K}$):', layout=slider_layout, style=style)
w_g_L = widgets.FloatSlider(value=0.3, min=0.0, max=2.0, step=0.1, description='Leak Conductance ($g_{L}$):', layout=slider_layout, style=style)

w_E_Na = widgets.FloatSlider(value=50.0, min=-50.0, max=100.0, step=5.0, description='Na+ Reversal Potential ($E_{Na}$):', layout=slider_layout, style=style)
w_E_K = widgets.FloatSlider(value=-77.0, min=-100.0, max=0.0, step=5.0, description='K+ Reversal Potential ($E_{K}$):', layout=slider_layout, style=style)
w_E_L = widgets.FloatSlider(value=-54.387, min=-80.0, max=-20.0, step=1.0, description='Leak Reversal Potential ($E_{L}$):', layout=slider_layout, style=style)

w_inj = widgets.FloatSlider(value=10.0, min=0.0, max=50.0, step=1.0, description='Injection Amplitude ($I_{inj}$):', layout=slider_layout, style=style)
w_Cm = widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1, description='Membrane Capacitance ($C_m$):', layout=slider_layout, style=style)

# Group the sliders vertically for the left panel
ui_controls = widgets.VBox([
    widgets.HTML("<b>Ion Channel Conductances</b>"), w_g_Na, w_g_K, w_g_L,
    widgets.HTML("<br><b>Reversal (Nernst) Potentials</b>"), w_E_Na, w_E_K, w_E_L,
    widgets.HTML("<br><b>Cellular & External Properties</b>"), w_Cm, w_inj
], layout=widgets.Layout(width='350px', padding='10px', border='solid 1px #ccc'))

# Link the sliders to the update function
interactive_plot = widgets.interactive_output(update_simulation, {
    'g_Na': w_g_Na, 'g_K': w_g_K, 'g_L': w_g_L,
    'E_Na': w_E_Na, 'E_K': w_E_K, 'E_L': w_E_L,
    'i_inj_amp': w_inj, 'C_m': w_Cm
})

# Display the sliders on the left (ui_controls) and the output/plots on the right (out)
display(widgets.HBox([ui_controls, out]))